<a href="https://colab.research.google.com/github/pastrop/kaggle/blob/master/RAG_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
%%capture
!pip install llama-index llama-index-llms-anthropic llama-index-embeddings-huggingface anthropic
pip install llama-index-packs-raptor umap-learn scikit-learn

In [22]:
"""
RAG with LlamaIndex + Claude + local embeddings.
- LLM:        Claude Sonnet 4.6 (Anthropic API) — used to synthesize answers
- Embeddings: BAAI/bge-small-en-v1.5 (local, free, runs on CPU)
- Loader:     SimpleDirectoryReader (handles PDF + Markdown)
- Citations:  CitationQueryEngine (inline [1][2] markers + source metadata)
"""

from __future__ import annotations

import os
import sys
from pathlib import Path

import anthropic
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.query_engine import CitationQueryEngine
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.anthropic import Anthropic
from llama_index.core.node_parser import MarkdownNodeParser, SentenceSplitter
from llama_index.readers.file import PDFReader
#Raptor Pack
from llama_index.packs.raptor import RaptorPack

# ---------- Config ----------
DOCS_DIR = "/content"                          # where your PDFs/MDs live
LLM_MODEL = "claude-sonnet-4-5"              # solid balance of cost/quality
EMBED_MODEL = "BAAI/bge-small-en-v1.5"       # local, free
TOP_K = 3                                    # how many chunks to retrieve


In [15]:
# ---------- 1. Validate the API key ----------
def validate_api_key() -> str:
    """Fail fast with a clear message if the key is missing, malformed, or invalid."""
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")

    if not api_key:
        sys.exit(
            "ERROR: ANTHROPIC_API_KEY is not set.\n"
            "  1. Get a key at https://console.anthropic.com/\n"
            "  2. Set it:  export ANTHROPIC_API_KEY='sk-ant-...'"
        )

    if not api_key.startswith("sk-ant-"):
        print(
            "WARNING: key doesn't start with 'sk-ant-'. "
            "Anthropic keys normally do. Continuing anyway..."
        )

    # Lightweight liveness check: smallest possible call to confirm the key works.
    # Costs a fraction of a cent and surfaces auth errors immediately instead of
    # after you've built the entire index.
    try:
        client = anthropic.Anthropic(api_key=api_key)
        client.messages.create(
            model="claude-haiku-4-5",  # cheapest model for the probe
            max_tokens=1,
            messages=[{"role": "user", "content": "ping"}],
        )
    except anthropic.AuthenticationError:
        sys.exit("ERROR: ANTHROPIC_API_KEY is invalid or has been revoked.")
    except anthropic.RateLimitError:
        print("WARNING: hit a rate limit on the key check — key is valid, continuing.")
    except anthropic.APIConnectionError as e:
        sys.exit(f"ERROR: could not reach Anthropic API: {e}")
    except Exception as e:
        # Non-fatal: don't block startup on unexpected probe failures
        print(f"WARNING: key liveness check failed unexpectedly: {e}")

    print("✓ Anthropic API key validated.")
    return api_key

In [16]:
# ---------- 2. Configure LlamaIndex globally ----------
def configure(api_key: str) -> None:
    Settings.llm = Anthropic(
        model=LLM_MODEL,
        api_key=api_key,
        max_tokens=1024,
    )
    # First run downloads the model (~133 MB) to ~/.cache/huggingface/
    Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL)
    print(f"✓ Configured: LLM={LLM_MODEL}, embeddings={EMBED_MODEL} (local)")


# ---------- 3. Load documents and build the index ----------
def build_index(docs_dir: str) -> VectorStoreIndex:
    path = Path(docs_dir)
    if not path.exists():
        sys.exit(f"ERROR: documents directory '{docs_dir}' not found.")

    # Load PDFs and MDs separately so we can apply the right parser to each.
    pdf_docs = SimpleDirectoryReader(
        input_dir=docs_dir, required_exts=[".pdf"],
        recursive=True, filename_as_id=True,
    ).load_data()

    md_docs = SimpleDirectoryReader(
        input_dir=docs_dir, required_exts=[".md"],
        recursive=True, filename_as_id=True,
    ).load_data()

    if not pdf_docs and not md_docs:
        sys.exit(f"ERROR: no .pdf or .md files found in '{docs_dir}'.")

    # Parsers
    md_parser = MarkdownNodeParser()                                # splits on # ## ### ...
    pdf_parser = SentenceSplitter(chunk_size=1024, chunk_overlap=200)  # sentence-aware

    md_nodes = md_parser.get_nodes_from_documents(md_docs)
    pdf_nodes = pdf_parser.get_nodes_from_documents(pdf_docs)
    all_nodes = md_nodes + pdf_nodes

    print(f"✓ Parsed {len(md_docs)} md → {len(md_nodes)} nodes, "
          f"{len(pdf_docs)} pdf → {len(pdf_nodes)} nodes. Building index...")

    return VectorStoreIndex(all_nodes, show_progress=True)


# ---------- 4. Query with inline citations ----------
def ask(index: VectorStoreIndex, question: str) -> None:
    query_engine = CitationQueryEngine.from_args(
        index,
        similarity_top_k=TOP_K,
        citation_chunk_size=512,
    )
    response = query_engine.query(question)

    print(f"\n=== Q: {question} ===")
    print(f"\nANSWER:\n{response.response}")

    print("\nSOURCES:")
    for i, node in enumerate(response.source_nodes, 1):
        meta = node.metadata
        fname = meta.get("file_name", "unknown")
        page = meta.get("page_label", "—")  # PDFs only; MDs won't have this
        score = node.score if node.score is not None else 0.0
        print(f"  [{i}] {fname}  (page: {page})  score={score:.3f}")

In [18]:
key = validate_api_key()

✓ Anthropic API key validated.


In [19]:
%%capture
configure(key)
index = build_index(DOCS_DIR)

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
ask(index, "can you pull me consensus opinion on the dow performance through the end of the year?")
#ask(index, "Who is responsible for incident response?")


=== Q: can you pull me consensus opinion on the dow performance through the end of the year? ===

ANSWER:
Based on the consensus forecasts as of May 11, 2026, with the Dow trading around 49,300, the **consensus base case is +5% to +9% to year-end** [2].

Here's the breakdown of year-end 2026 forecasts from various sources [3]:

- **Trading Economics (bear case):** ~42,600 (-13%)
- **LongForecast:** ~53,600 (+9%)
- **WalletInvestor:** ~51,300 (+4%)
- **Meyka:** ~52,600 (+7%)
- **Yardeni / CNBC strategist consensus:** 52,000–53,000 (+5% to +7%)
- **Bullish (CoinPriceForecast):** ~54,700 (+11%)
- **Most bullish (cyclical model):** ~64,500 (+31%)

The consensus year-end projections cluster roughly in the $47,000–$54,000 range [1]. Key drivers analysts are watching include S&P 500 earnings growth expectations of roughly 15% in 2026, tariff impacts on profit margins, and the macro environment which is expected to remain unstable given policy crosscurrents and a wobbly labor market [1].

SOU

=== Q: can you pull me consensus opinion on the dow performance through the end of the year? ===

ANSWER:
Based on the consensus forecasts as of May 11, 2026, with the Dow trading around 49,300, the **consensus base case is +5% to +9% to year-end** [2].

Here's the breakdown of year-end 2026 forecasts from various sources [3]:

- **Trading Economics (bear case):** ~42,600 (-13%)
- **LongForecast:** ~53,600 (+9%)
- **WalletInvestor:** ~51,300 (+4%)
- **Meyka:** ~52,600 (+7%)
- **Yardeni / CNBC strategist consensus:** 52,000–53,000 (+5% to +7%)
- **Bullish (CoinPriceForecast):** ~54,700 (+11%)
- **Most bullish (cyclical model):** ~64,500 (+31%)

The consensus year-end projections cluster roughly in the $47,000–$54,000 range [1]. Key drivers analysts are watching include S&P 500 earnings growth expectations of roughly 15% in 2026, tariff-related impacts on profit margins, and the macro environment which is expected to remain unstable given policy crosscurrents and a wobbly labor market [1].

SOURCES:
  [1] dow_forecast_portfolio_deployment.md  (page: —)  score=0.803
  [2] dow_forecast_portfolio_deployment.md  (page: —)  score=0.803
  [3] dow_forecast_portfolio_deployment.md  (page: —)  score=0.771
  [4] dow_forecast_portfolio_deployment.md  (page: —)  score=0.771
  [5] dow_forecast_portfolio_deployment.md  (page: —)  score=0.771
  [6] dow_forecast_portfolio_deployment.md  (page: —)  score=0.729
  [7] dow_forecast_portfolio_deployment.md  (page: —)  score=0.729

# Raptor Pipeline

In [ ]:
def build_raptor_pipeline(all_nodes):
    """Builds a RAPTOR retriever in-memory, reusing the same LLM and embed model.
    Returns a RetrieverQueryEngine so it's a drop-in alternative to your flat one."""
    from llama_index.core.query_engine import RetrieverQueryEngine

    print("Building RAPTOR — this calls Claude to summarize clusters (minutes, costs $).")
    pack = RaptorPack(
        all_nodes,
        embed_model=Settings.embed_model,
        llm=Settings.llm,
        similarity_top_k=TOP_K,   # or whatever constant you already use
        mode="collapsed",
    )
    return RetrieverQueryEngine.from_args(pack.retriever)

In [ ]:
raptor_qe = build_raptor_pipeline(all_nodes)

In [ ]:
question = "can you pull me consensus opinion on the dow performance through the end of the year?"
raptor_resp = raptor_qe.query(question)
print(raptor_resp.response)
for n in raptor_resp.source_nodes:
    print(n.metadata.get("file_name"), n.metadata.get("page_label"), n.score)

# Outakes

In [17]:
#standard index no md specific processing
def build_index(docs_dir: str) -> VectorStoreIndex:
    path = Path(docs_dir)
    if not path.exists():
        sys.exit(f"ERROR: documents directory '{docs_dir}' not found.")

    documents = SimpleDirectoryReader(
        input_dir=docs_dir,
        required_exts=[".pdf", ".md"],
        recursive=True,
        filename_as_id=True,    # stable IDs based on filename
    ).load_data()

    if not documents:
        sys.exit(f"ERROR: no .pdf or .md files found in '{docs_dir}'.")

    print(f"✓ Loaded {len(documents)} document chunk(s). Building index...")
    # All embedding happens locally here — no API calls.
    return VectorStoreIndex.from_documents(documents, show_progress=True)